Initial Logging

In [1]:
import sys
sys.path.append("..")  # if notebook is in notebooks/, adjust so src/ is importable

from src.utils.config_loader import load_config
config = load_config()
print(config)

{'paths': {'tesseract_cmd': 'C:\\Program Files\\Tesseract-OCR\\tesseract.exe', 'raw_data_dir': 'data/raw', 'processed_data_dir': 'data/processed'}, 'extraction': {'supported_formats': ['.pdf', '.docx', '.txt', '.jpg', '.jpeg', '.png'], 'ocr_dpi': 300, 'min_text_length_for_native_pdf': 20}, 'preprocessing': {'spacy_model': 'en_core_web_sm', 'min_token_length': 2, 'use_lemmatization': True, 'use_stemming': False, 'custom_stopwords': ['exam', 'answer', 'question', 'marks', 'page'], 'protected_terms': ['h2o', 'co2', 'dna', 'rna', 'ph'], 'preserve_numbers': True}, 'similarity': {'method': 'tfidf', 'semantic_model': 'all-MiniLM-L6-v2', 'thresholds': {'excellent': 0.75, 'good': 0.5, 'poor': 0.0}, 'tfidf_ngram_range': [1, 2]}, 'reporting': {'output_dir': 'data/outputs', 'top_n_matching_sentences': 3, 'chart_dpi': 150, 'wordcloud_max_words': 100}}


In [2]:
from src.utils.logger import get_logger
logger = get_logger("test")
logger.info("Logger test message")
logger.warning("Logger warning test")

2026-08-02 23:42:44 | INFO     | test | Logger test message
2026-08-02 23:42:44 | WARNING  | test | Logger warning test


In [3]:
import fitz, docx, pytesseract, cv2, pdf2image
from pdf2image import convert_from_path
from PIL import Image
print("All extraction dependencies imported successfully")

All extraction dependencies imported successfully


Files Extraction Tests

In [4]:
from src.extraction.pdf_extractor import extract_text_from_pdf

result = extract_text_from_pdf("../data/samples/ANS_PDF.pdf")
print("Metadata:", result["metadata"])
print("\n--- Text preview (first 300 chars) ---")
print(result["text"][:300])

2026-08-02 23:42:58 | INFO     | src.extraction.pdf_extractor | Attempting to extract text from PDF: ../data/samples/ANS_PDF.pdf


Metadata: {'filename': '../data/samples/ANS_PDF.pdf', 'page_count': 1, 'char_count': 752}

--- Text preview (first 300 chars) ---
Exam Answer: Cell Membrane Structure and
Function
Student: Variant B (PDF submission)
The plasma membrane, also called the cell membrane, is a flexible boundary that encloses all living
cells. It mainly consists of a phospholipid bilayer along with embedded proteins, cholesterol, and
carbohydrate mo


In [5]:
from src.extraction.docx_extractor import extract_text_from_docx

result = extract_text_from_docx("../data/samples/ANS_DOCX.docx")
print("Metadata:", result["metadata"])
print("\n--- Text preview ---")
print(result["text"][:300])

2026-08-02 23:42:58 | INFO     | src.extraction.docx_extractor | Attempting to extract text from DOCX: ../data/samples/ANS_DOCX.docx


Metadata: {'filename': '../data/samples/ANS_DOCX.docx', 'char_count': 850}

--- Text preview ---
Exam Answer: Cell Membrane Structure and Function
Student: Variant A (DOCX submission)
The cell membrane, also known as the plasma membrane, is a thin, flexible barrier that surrounds every living cell. It is composed mainly of a phospholipid bilayer embedded with proteins, cholesterol, and carbohyd


In [6]:
from src.extraction.txt_extractor import extract_text_from_txt

result = extract_text_from_txt("../data/samples/ANS_TXT.txt")
print("Metadata:", result["metadata"])
print("\n--- Text preview ---")
print(result["text"][:300])

2026-08-02 23:42:59 | INFO     | src.extraction.txt_extractor | Attempting to extract text from TXT: ../data/samples/ANS_TXT.txt


Metadata: {'filename': '../data/samples/ANS_TXT.txt', 'char_count': 734}

--- Text preview ---
Exam Answer: Cell Membrane Structure and Function
Student: Variant C (TXT submission)

Cells are the basic unit of life and are surrounded by a membrane made of lipids. The cell membrane helps protect the cell from its environment. Some proteins are found within the membrane which help transport cer


In [7]:
from src.extraction.ocr_extractor import extract_text_from_image

result = extract_text_from_image("../data/samples/ANS_PNG.png")
print("Metadata:", result["metadata"])
print("\n--- Text preview ---")
print(result["text"][:300])

2026-08-02 23:42:59 | INFO     | src.extraction.ocr_extractor | Extracting text via OCR from: ../data/samples/ANS_PNG.png
2026-08-02 23:42:59 | INFO     | src.extraction.ocr_extractor | Preprocessed image: ../data/samples/ANS_PNG.png


Metadata: {'filename': '../data/samples/ANS_PNG.png', 'char_count': 1007}

--- Text preview ---
Exam Answer: Cell Membrane Structure and Function
Student: Variant D (PNG submission)

The cellular membrane serves as a dynamic and protective outer boundary for all living cells. Structurally, it
is constructed from a fluid lipid bilayer that actively incorporates various proteins, carbohydrates, 


Full Dispatcher Batch Test

In [8]:
import os
from src.extraction.extractor import extract_text

sample_dir = "../data/samples"
for fname in os.listdir(sample_dir):
    path = os.path.join(sample_dir, fname)
    result = extract_text(path)
    status = "ERROR" if "error" in result["metadata"] else "OK"
    print(f"{fname:30s} -> {status:6s} | char_count={result['metadata'].get('char_count', 0)}")

2026-08-02 23:43:06 | INFO     | src.extraction.docx_extractor | Attempting to extract text from DOCX: ../data/samples\ANS_DOCX.docx
2026-08-02 23:43:06 | INFO     | src.extraction.pdf_extractor | Attempting to extract text from PDF: ../data/samples\ANS_PDF.pdf
2026-08-02 23:43:06 | INFO     | src.extraction.ocr_extractor | Extracting text via OCR from: ../data/samples\ANS_PNG.png
2026-08-02 23:43:06 | INFO     | src.extraction.ocr_extractor | Preprocessed image: ../data/samples\ANS_PNG.png


ANS_DOCX.docx                  -> OK     | char_count=850
ANS_PDF.pdf                    -> OK     | char_count=752


2026-08-02 23:43:18 | INFO     | src.extraction.txt_extractor | Attempting to extract text from TXT: ../data/samples\ANS_TXT.txt


ANS_PNG.png                    -> OK     | char_count=1007
ANS_TXT.txt                    -> OK     | char_count=734
